In [1]:
# 02_preprocessing_airbnb.ipynb - KESİN ÇÖZÜM
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

print("🚀 Preprocessing gestartet...")

# Daten laden
data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "AB_NYC_2019.csv")
df = pd.read_csv(data_path)
print(f"📦 Datensatz geladen: {df.shape}")

# Grundreinigung
df = df.drop(columns=["id", "name", "host_name", "last_review"], errors="ignore")
df.drop_duplicates(inplace=True)
df = df[(df["price"] > 0) & (df["price"] < 1000)]
print(f"📊 Bereinigter Datensatz: {df.shape}")

# Feature Selection
features = ["neighbourhood_group", "room_type", "latitude", "longitude", 
           "minimum_nights", "number_of_reviews", "price"]
df = df[features].dropna()

# Label Encoding für kategorische Variablen
print("🔤 Kodiere kategorische Variablen...")
le_neighbourhood = LabelEncoder()
le_room = LabelEncoder()

df["neighbourhood_group_encoded"] = le_neighbourhood.fit_transform(df["neighbourhood_group"])
df["room_type_encoded"] = le_room.fit_transform(df["room_type"])

# Finale Features
X = df[["neighbourhood_group_encoded", "room_type_encoded", "latitude", "longitude", 
       "minimum_nights", "number_of_reviews"]]
y = df["price"]

# Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 Train-Test-Split:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")

# Datenüberprüfung
print("\n🔍 Datenüberprüfung:")
print("X_train Datentypen:")
print(X_train.dtypes.value_counts())

# Sicherstellen, dass alle Daten numerisch sind
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        print(f"❌ WARNUNG: {col} ist immer noch kategorisch!")
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# NaN-Werte auffüllen
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print("\n✅ Finale Datentypen:")
print("X_train:", X_train.dtypes.value_counts())

# Daten speichern
save_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
os.makedirs(save_path, exist_ok=True)

X_train.to_csv(os.path.join(save_path, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(save_path, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(save_path, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(save_path, "y_test.csv"), index=False)

# Encoder speichern
models_path = os.path.join(os.path.dirname(os.getcwd()), "models")
os.makedirs(models_path, exist_ok=True)
joblib.dump({'neighbourhood': le_neighbourhood, 'room_type': le_room}, 
            os.path.join(models_path, "label_encoders.pkl"))

print(f"\n💾 Daten gespeichert unter: {save_path}")
print("💾 Encoder gespeichert unter: ../models/label_encoders.pkl")

print("\n🎯 PREPROCESSING ERFOLGREICH ABGESCHLOSSEN!")
print(f"• Trainingsdaten: {X_train.shape}")
print(f"• Testdaten: {X_test.shape}")

🚀 Preprocessing gestartet...


📦 Datensatz geladen: (48895, 16)
📊 Bereinigter Datensatz: (48586, 12)
🔤 Kodiere kategorische Variablen...
📊 Train-Test-Split:
   X_train: (38868, 6)
   X_test: (9718, 6)

🔍 Datenüberprüfung:
X_train Datentypen:
int64      4
float64    2
Name: count, dtype: int64

✅ Finale Datentypen:
X_train: int64      4
float64    2
Name: count, dtype: int64



💾 Daten gespeichert unter: C:\Users\okans\OneDrive\Pictures\Masaüstü\airbnb-data-analysis\data\processed
💾 Encoder gespeichert unter: ../models/label_encoders.pkl

🎯 PREPROCESSING ERFOLGREICH ABGESCHLOSSEN!
• Trainingsdaten: (38868, 6)
• Testdaten: (9718, 6)
